# A tour of the `oif` package

The [main tutorial](objects_in_focus_tutorial.ipynb) walks one analysis from
start to finish. This notebook is the other kind of guide: **one section per
module**, showing what each function is for and how to call it. Skim it top to
bottom once, then come back to the section you need.

| module | what it holds |
|---|---|
| `oif.paths` | find and validate a data tree |
| `oif.datasets` | `OiF` and `Scene` — the high-level way in |
| `oif.annotations` | CVAT polygon XML → objects |
| `oif.masks` | polygons ↔ label maps, label recovery |
| `oif.depth` | MonoDepth2 disparity → usable depth |
| `oif.fixations` | read and clean eye-movement data |
| `oif.mapping` | fixations → objects |
| `oif.features` | size, eccentricity, depth, salience per object |
| `oif.model` | the object-based attention model |
| `oif.salience` | salience maps and the centre-bias baseline |
| `oif.viz` | every plot in this notebook |

Run the cells in order — later sections reuse objects made in earlier ones.

## Setup

Install the package and fetch six scenes (images, annotations, depth maps and
the real fixations). Skip the download cell if you are running inside a full
clone of the repository.

In [ ]:
!pip install -q "git+https://github.com/ehhall/objects-in-focus.git"

import oif
print("oif version", oif.__version__)

In [ ]:
import urllib.request
from pathlib import Path

SCENES = ["target_livingroom_IDS01", "target_bakery", "target_kitchen_IDS01",
          "target_garden", "target_subway", "target_beach"]
RAW = "https://raw.githubusercontent.com/ehhall/objects-in-focus/main"
root = Path("objects-in-focus")
for folder in ("images", "annotations", "depth", "raw"):
    (root / folder).mkdir(parents=True, exist_ok=True)

def fetch(url, dest):
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)

for s in SCENES:
    fetch(f"{RAW}/images/{s}.png",       root / "images" / f"{s}.png")
    fetch(f"{RAW}/annotations/{s}.xml",  root / "annotations" / f"{s}.xml")
    fetch(f"{RAW}/depth/{s}_disp.npy",   root / "depth" / f"{s}_disp.npy")
    fetch(f"{RAW}/raw/{s}_memorize.npy", root / "raw" / f"{s}_memorize.npy")
print("fetched", len(SCENES), "scenes")

## `oif.paths` — where is the data, and is it complete?

`find_data_root` searches an explicit path, then `$OIF_DATA_ROOT`, then the
working directory and its parents, for a folder holding `images/` and
`annotations/`. `DataRoot` wraps that folder with case-insensitive file
lookup (two scenes are spelled inconsistently in the release) and an
integrity check.

In [ ]:
from oif import DataRoot, find_data_root

print("root found:", find_data_root(root))

dr = DataRoot(root)
print(len(dr), "scenes;", "first three:", dr.scenes()[:3])
print("one resolved path:", dr.path("images", "target_bakery"))
print()
print(dr.check())          # missing / truncated / case-mismatched files

## `oif.datasets` — `OiF` and `Scene`, the high-level way in

Almost everything in the package is reachable from these two. `OiF` is the
dataset; indexing it gives a `Scene`, and every array on a scene is loaded
lazily the first time you touch it.

In [ ]:
from oif import OiF

data = OiF(root)
scene = data["target_livingroom_IDS01"]     # by name (case-insensitive), or data[0]

print(scene)
print("image      ", scene.image.shape)          # (768, 1024, 3) uint8
print("label_map  ", scene.label_map.shape)      # object id per pixel, 0 = background
print("depth      ", scene.depth.shape)          # 0 near ... 1 far
print("labels     ", dict(list(scene.labels.items())[:4]))
print("object_ids ", scene.object_ids[:8])
print("fixations  ", len(scene.fixations()), "recorded while memorizing")

In [ ]:
# dataset-level helpers
print(data.summary().head(3), "\n")             # per-scene object counts
print(data.stats())                              # the numbers a Stimuli section needs

## `oif.annotations` — the polygons as drawn

Each scene's CVAT XML export becomes a `SceneAnnotation`: a list of
`Polygon`s with a label, an occlusion flag and their vertex coordinates.
`objects()` and `background()` split them by label — sky, wall, floor and
friends are surfaces, not objects.

In [ ]:
from oif import read_annotation

ann = read_annotation(root / "annotations" / "target_livingroom_IDS01.xml")
print(ann)
print("objects:   ", [p.label for p in ann.objects()][:8], "...")
print("background:", [p.label for p in ann.background()])

poly = ann.objects()[0]
print(f"\nfirst polygon: {poly.label!r}, {poly.n_vertices} vertices, "
      f"bbox {tuple(round(v) for v in poly.bbox)}, occluded={poly.occluded}")

## `oif.masks` — polygons ↔ label maps

`build_label_map` paints the polygons **back to front** (occlusion flags
first, then depth), so the nearer object owns every contested pixel — that is
the object the viewer actually saw. `recover_labels` runs the other
direction: given a label map of anonymous integers, it works out which
polygon each id was painted from.

In [ ]:
from oif import build_label_map, recover_labels, region_ids
from oif.masks import region_areas, resize_label_map, split_components
import numpy as np

label_map, painted_order = build_label_map(scene.annotation, scene.depth)
print("ids in the map:", region_ids(label_map)[:8], "...")
print("largest object:", max(region_areas(label_map).items(), key=lambda kv: kv[1]))

recovered = recover_labels(label_map, scene.annotation)
print("recovered:", [(r.mask_id, r.label, round(r.score, 3)) for r in recovered[:4]])

small = resize_label_map(label_map, (192, 256))   # nearest-neighbour, ids intact
print("resized:", small.shape, "same ids:", set(np.unique(small)) == set(np.unique(label_map)))

blobs = split_components(label_map == recovered[0].mask_id, min_size=10)
print(f"connected parts of {recovered[0].label!r}:", len(set(np.unique(blobs)) - {0}))

## `oif.depth` — disparity in, depth out

The release ships raw MonoDepth2 disparity (bigger = nearer, arbitrary
units, low resolution). `load_depth` / `to_depth` resize to the image,
invert, and rescale to \[0, 1\] so that **bigger = farther** and `log1p`
is safe. Pass `invert=False, rescale=False` for the untouched array.

In [ ]:
from oif.depth import load_depth, to_depth

path = root / "depth" / "target_livingroom_IDS01_disp.npy"
depth = load_depth(path, shape=(768, 1024))
raw   = load_depth(path, shape=(768, 1024), invert=False, rescale=False)
print(f"usable depth: {depth.min():.2f} (near) to {depth.max():.2f} (far)")
print(f"raw disparity: {raw.min():.1f} to {raw.max():.1f} (bigger = nearer)")

## `oif.fixations` — reading and cleaning eye-movement data

Everything downstream expects one canonical table:
`subject, image, fix_index, x, y, duration, task`. Three ways to get it:

- `load_fixations(folder)` — reads **every** file in a folder: the released
  binary `.npy` maps, DataViewer CSV exports, the older
  `locs_1`/`locs_2`/`durs` layout, or your own columns via `columns={...}`.
- `load_fixation_map(path)` + `map_to_fixations(arr, image, task)` — one
  released map, by hand.
- `scene.fixations()` / `data.fixations()` — the same, from a `Scene` or the
  whole dataset.

In [ ]:
from oif import load_fixation_map, load_fixations, map_to_fixations, filter_fixations
from oif.fixations import summarize

arr = load_fixation_map(root / "raw" / "target_livingroom_IDS01_memorize.npy")
print("one map:", arr.shape, "-", int(arr.sum()), "fixations")
print(map_to_fixations(arr, "target_livingroom_IDS01", "memorize").head(3), "\n")

fixations = load_fixations(root / "raw")          # every file in the folder
fixations = filter_fixations(fixations)           # 50-1500 ms, drop first, on-image
                                                  # (no-ops where a column is empty)
print(summarize(fixations))

In [ ]:
# fixations as arrays: accumulate, then blur with the Torralba low-pass filter
from oif import fixation_array, gaussian_blur, density_map

one_scene = fixations[fixations["image"] == scene.name]
counts = fixation_array(one_scene, shape=scene.shape)      # spikes at fixated pixels
smooth = density_map(one_scene, shape=scene.shape, fc=6)   # = gaussian_blur(counts)
print("array:", counts.shape, "- total", int(counts.sum()), "- blurred max", round(float(smooth.max()), 4))

## `oif.mapping` — which object did each fixation land on?

`map_fixations` adds `mask_id` and `label` columns to a fixation table.
Three assignment methods:

| method | what it does |
|---|---|
| `"point"` | the object under the fixation pixel |
| `"disc"` | the object covering most of a disc of `radius` px — absorbs calibration drift |
| `"nearest"` | as point, but background fixations snap to the nearest object within `radius` |

Fixations on nothing come back as `mask_id 0`, label `"background"` — never
silently dropped. `object_fixations` then aggregates to one row per object,
keeping the objects nobody looked at.

In [ ]:
from oif import map_fixations, object_fixations, assign_fixations
import numpy as np

mapped = map_fixations(one_scene, scene.label_map, scene.labels, method="disc", radius=25)
print(mapped[["x", "y", "mask_id", "label"]].head(4), "\n")

per_object = object_fixations(mapped, scene.label_map, scene.labels, scene=scene.name)
print(per_object.sort_values("n_fixations", ascending=False).head(4), "\n")

# the array-level core, if you have bare coordinates
ids = assign_fixations(np.array([512.0]), np.array([384.0]), scene.label_map,
                       method="disc", radius=25)
print("dead centre of this scene lands on:", scene.labels.get(int(ids[0]), "background"))

In [ ]:
# two more mapping utilities
from oif import fill_objects, object_value_map
from oif.mapping import rescale_fixations

# paint a per-object number back onto the image (used by viz.show_object_values)
painted = fill_objects(scene.label_map, dict(zip(per_object["mask_id"], per_object["n_fixations"])))
print("painted:", painted.shape, "max", painted.max())

# summarise any pixel map per object (mean/max/sum/center)
mean_depth = object_value_map(scene.label_map, scene.depth, stat="mean")
first = int(scene.object_ids[0])
print(f"mean depth of {scene.labels[first]!r}:", round(mean_depth[first], 3))

# display coordinates -> image coordinates (letterboxed displays use mode='fit')
demo = one_scene.head(2).copy()
print(rescale_fixations(demo, from_shape=(1050, 1680), to_shape=(768, 1024), mode="fit")[["x", "y"]])

## `oif.features` — what the model sees

`object_features` measures every object in a label map: visible **size** in
pixels, **eccentricity** of its centre, **depth** and **salience** sampled at
its centre of mass (or summarised over the object — `depth_stat="mean"`).
`add_model_terms` then builds the model's predictors: `log_size`,
`log_depth`, `z_ecc`, `z_salience` and the `log_sum` outcome.

In [ ]:
from oif import object_features, add_model_terms

features = object_features(scene.label_map, depth=scene.depth,
                           labels=scene.labels, scene=scene.name)
print(features.head(4), "\n")

# scene.object_table() = features + fixation counts in one call
table = data.object_tables(fixations, method="disc", radius=25)
table = add_model_terms(table)
print(table[["label", "size", "log_size", "z_ecc", "n_fixations", "log_sum"]].head(4))

## `oif.model` — the object-based attention model

`log1p(fixations) ~ log1p(size) + log1p(depth) + z(ecc) + z(salience)`,
plain least squares. Terms with no data (here: salience) are dropped with a
warning. `cross_validate_by_scene` holds out whole **scenes** — objects in a
scene share viewers and geometry, so object-level splits would leak.
`PUBLISHED_FIT` holds the paper's numbers for comparison.

In [ ]:
from oif import ObjectAttentionModel, PUBLISHED_FIT
from oif.model import cross_validate_by_scene
import pandas as pd

model = ObjectAttentionModel().fit(table)
print(model.coefficients.round(3), "\n")
print({k: round(v, 3) for k, v in model.score(table).items()}, "\n")

table["predicted"] = model.predict(table, scale="count")   # or scale="log"
print(cross_validate_by_scene(table, n_folds=3).round(3), "\n")
print(pd.DataFrame(PUBLISHED_FIT).T[["n_objects", "r2"]])

## `oif.salience` — maps, and the baseline to beat

Salience maps are not bundled (generate them with
[DeepGaze](https://github.com/matthias-k/DeepGaze), one `.npy` per scene,
then `load_salience_maps("salience/")`). What *is* built in: `center_bias`,
the Gaussian every salience model must beat, and `salience_per_object` to
read any pixel map off per object.

In [ ]:
from oif.salience import center_bias, salience_per_object

bias = center_bias(scene.shape)
per_obj = salience_per_object(scene.label_map, bias)
central = max(per_obj, key=per_obj.get)
print("most central object by centre-bias:", scene.labels.get(central))

## `oif.viz` — every picture in one cell

All plotting lives here; each function takes an optional `ax`. `matplotlib`
is only imported when you plot, so the rest of the package runs without it.

In [ ]:
import matplotlib.pyplot as plt
from oif.viz import (show_scene, show_objects, show_fixations,
                     show_object_values, show_density, outline_objects)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
show_scene(scene.image, ax=axes[0, 0], title="show_scene")
show_objects(scene.image, scene.label_map, ax=axes[0, 1], title="show_objects")
show_fixations(scene.image, one_scene, ax=axes[0, 2], size=14, title="show_fixations")
show_object_values(scene.label_map,
                   dict(zip(per_object["mask_id"], per_object["n_fixations"])),
                   ax=axes[1, 0], cmap="magma", title="show_object_values", colorbar=False)
show_density(scene.density(one_scene), ax=axes[1, 1], title="show_density")
axes[1, 2].imshow(outline_objects(scene.image, scene.label_map, thickness=3))
axes[1, 2].set_title("outline_objects (returns an array)")
axes[1, 2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# the dataset's contact sheet
from oif.viz import scene_grid
scene_grid((data[s].image for s in data.scene_names), rows=2, cols=3,
           titles=data.scene_names, figsize=(15, 7))
plt.show()

## The command line

Everything above has a no-code counterpart (`--root` if the data lives
elsewhere):

```bash
oif check                    # paths: is my copy complete?
oif stats                    # datasets: the headline numbers
oif labels                   # masks: write derived/labels.csv
oif repair                   # masks: rebuild the truncated mask files
oif fixations                # fixations: read + clean + summarise raw/
oif objects --fixations raw/ --method disc --radius 25 --out objects.csv
oif demo target_bakery --labels
```

## Where next

- [The main tutorial](objects_in_focus_tutorial.ipynb) — one full analysis, start to finish
- [`docs/api.md`](../docs/api.md) — this notebook as a reference card
- [`docs/dataset.md`](../docs/dataset.md) — every file format in detail
- [`docs/coco.md`](../docs/coco.md) — `oif.datasets.COCOFreeview`, the same
  analysis on MS-COCO scenes (needs the COCO-Freeview data, so it is not
  demonstrated here)